# Cosmx Data from Multiple Small Lung Carcinoma Patients

As Processed in [NicheCompass](https://www.nature.com/articles/s41588-025-02120-6#code-availability), and downloaded from their [Gdrive](https://drive.google.com/drive/folders/1sqoqCq1y5NMIbC1K7uq6v4PBWDPQQJgY).

In [1]:
import scanpy as sc
import numpy as np
from cellina._spatial_utils import weighted_pseudobulks, spatial_neighbors
import pathlib
from scipy.sparse import csr_matrix

/data/a330d/miniforge3/envs/cellina-reproduce/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Params

In [10]:
celltype_key = 'cell_type'
dataset_name = 'lung_cosmx'
batch_key = "batch"
sample_key = batch_key
niche_key = 'niche'

# Split
train_frac = 0.8
n_holdout = 2
n_val = False
test_samples = ['lung13', 'lung12']
K_NN = 50

#data_dir = pathlib.Path("../../data/nanostring_cosmx_human_nsclc/")
base_path = "/data2/a330d/datasets/cosmx/lung"
data_dir = pathlib.Path(f"{base_path}/nanostring_cosmx_human_nsclc/")

## Concat to a single AnnData object

In [3]:
h5ad_files = list(data_dir.glob("*.h5ad"))
adatas = []
for f in h5ad_files:
    adata = sc.read_h5ad(f)

    spatial_neighbors(adata, bandwidth=np.inf, max_neighbours=K_NN, standardize=False)
    adata.obs['neighbor'] = adata.obsp['spatial_connectivities'][:,0].toarray().astype(np.float32)

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.obsm['spatial_x'] = adata.obsp['spatial_connectivities'] @ adata.X / K_NN
    adata.obsm['spatial_x'] = csr_matrix(adata.obsm['spatial_x']).astype(np.float32)
        
    adatas.append(adata)

In [4]:
adata = sc.concat(adatas)

/data/a330d/miniforge3/envs/cellina-reproduce/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [5]:
adata

AnnData object with n_obs × n_vars = 702199 × 960
    obs: 'cell_ID', 'patient', 'batch', 'fov', 'cell_type', 'niche', 'cell_type_original', 'n_counts', 'neighbor'
    obsm: 'spatial', 'spatial_x'
    layers: 'counts'

In [6]:
adata.X = adata.layers['counts'].copy()

In [7]:
adata.X.max()

np.float32(365.0)

## Save Needed Params

In [8]:
adata.uns['default_params'] = {
    # Data Specific Parameters
    "dataset_name": "cosmx_melanoma",
    "K_NN": K_NN,
    "celltype_key": celltype_key,
    "niche_key": niche_key,
    "sample_key": sample_key,
    "batch_key": batch_key,
    # Split Parameters
    "n_holdout": n_holdout,
    "n_val_samples": False,
    "test_samples": test_samples,
    "train_frac": train_frac
}

In [9]:
# NOTE: needed since .index is duplicated
adata.obs.index = adata.obs['cell_ID']

In [12]:
#adata.write_h5ad("../../data/lung_cosmx.h5ad")
adata.write_h5ad(f"{base_path}/lung_cosmx.h5ad")